C:\Users\Administrator\AppData\Local\Programs\Python\Python39\lib\site-packages\huggingface_hub\file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


config.json:   0%|          | 0.00/688 [00:00<?, ?B/s]

模型类型： roberta
模型配置： RobertaConfig {
  "_name_or_path": "roberta-large-mnli",
  "_num_labels": 3,
  "architectures": [
    "RobertaForSequenceClassification"
  ],
  "attention_probs_dropout_prob": 0.1,
  "bos_token_id": 0,
  "classifier_dropout": null,
  "eos_token_id": 2,
  "hidden_act": "gelu",
  "hidden_dropout_prob": 0.1,
  "hidden_size": 1024,
  "id2label": {
    "0": "CONTRADICTION",
    "1": "NEUTRAL",
    "2": "ENTAILMENT"
  },
  "initializer_range": 0.02,
  "intermediate_size": 4096,
  "label2id": {
    "CONTRADICTION": 0,
    "ENTAILMENT": 2,
    "NEUTRAL": 1
  },
  "layer_norm_eps": 1e-05,
  "max_position_embeddings": 514,
  "model_type": "roberta",
  "num_attention_heads": 16,
  "num_hidden_layers": 24,
  "pad_token_id": 1,
  "position_embedding_type": "absolute",
  "transformers_version": "4.41.2",
  "type_vocab_size": 1,
  "use_cache": true,
  "vocab_size": 50265
}



In [12]:
from transformers import pipeline

# 加载零样本分类管道
classifier = pipeline("text-classification", model="FacebookAI/roberta-large-mnli")

# 指定候选标签
candidate_labels =['entailment', 'contradiction','neutral']
# 函数
def nli_predict(premise, hypothesis):

    # 将前提和假设拼接，使用分隔符来区分
    input_text = " premise:"+premise + " </s></s> "+"hypothesis:"+ hypothesis
    # 获取预测结果
    result = classifier(input_text)
    return result

# 测试
premise = "All birds can fly."
hypothesis = "Penguins cannot fly."
result = nli_predict(premise, hypothesis)
print(result)

Some weights of the model checkpoint at FacebookAI/roberta-large-mnli were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


[{'label': 'CONTRADICTION', 'score': 0.9789014458656311}]


In [13]:
import json
import torch
import numpy as np
import time
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForSequenceClassification


# 2. 从 JSON 文件加载测试数据
with open(r'D:\PycharmProjects\TDFilter\experiment5\qwen\radiation\basic\results_processed.jsonl', 'r',encoding='utf-8') as f:
    test_data = json.load(f)

# 3. 提取 generateKnowledge 和 flag 字段
generate_knowledge_texts = [item["knowledge"] for item in test_data]
true_flags = [item["flag"] for item in test_data]  # 提取真实的 flag 值
match_knowledge_texts = [item["matchingKnowledge"] for item in test_data]
confidence_flags=[item["confidence_flag"] for item in test_data]
score_of_confidences=[item["score_of_confidence"] for item in test_data]
# 4. 定义批处理大小和结果列表
batch_size = 64  # 可以根据你的硬件情况调整批次大小
predicted_classes = []
confidence_scores = []

# 定义输出文件路径
output_file = r'D:\PycharmProjects\TDFilter\experiment5\qwen\radiation\basic\results_processed_predict.jsonl'  # 使用 .jsonl 扩展名表示 JSON Lines 格式

# 5. 分批处理数据
for i in tqdm(range(0, len(generate_knowledge_texts), batch_size)):
    batch_texts = generate_knowledge_texts[i:i + batch_size]
    batch_labels = true_flags[i:i + batch_size]
    batch_match = match_knowledge_texts[i:i + batch_size]
    batch_confidence_flags = confidence_flags[i:i + batch_size]
    batch_score_of_confidences = score_of_confidences[i:i + batch_size]
    # 对当前批次的文本进行分词处理
    # inputs = tokenizer(batch_texts, padding=True, truncation=True, return_tensors="pt")

    # 使用模型进行推理
    # with torch.no_grad():
    #     outputs = model(**inputs)

    # 获取 logits 并计算 softmax 概率
    # logits = outputs.logits
    # probabilities = torch.softmax(logits, dim=-1)

    # 获取每个预测的类别和对应的概率（可信度）
    # batch_predictions = torch.argmax(logits, dim=-1).tolist()
    # batch_confidences = [prob[cls].item() for prob, cls in zip(probabilities, batch_predictions)]

    # predicted_classes.extend(batch_predictions)
    # confidence_scores.extend(batch_confidences)


    # 将预测结果、可信度、knowledge和真实标签以 JSON 格式写入文件
    with open(output_file, 'a', encoding='utf-8') as f_output:
        for knowledge, matchKnowledge,confidence_flag,score_of_confidence, true_flag in zip(batch_texts,batch_match, batch_confidence_flags ,batch_score_of_confidences,batch_labels):
            # 进行矛盾判断
            contradict_predict = nli_predict(premise=knowledge,hypothesis=matchKnowledge)
            if contradict_predict[0]['label'] == 'CONTRADICTION':
                contradict_flag = 0
            if contradict_predict[0]['label'] == 'ENTAILMENT':
                contradict_flag = 1
            if contradict_predict[0]['label'] == 'NEUTRAL':
                contradict_flag = -1
            contradict_score = contradict_predict[0]['score']
            json_line = json.dumps({
                'knowledge': knowledge,  # 保存 knowledge 字段
                'matchingKnowledge': matchKnowledge,  # 保存 matchingKnowledge 字段
                'confidence_flag': confidence_flag,
                'score_of_confidence': round(score_of_confidence, 2),  # 以小数形式保留两位
                'contradict_flag': contradict_flag,
                'score_of_contradict': round(contradict_score, 2),
                'true_flag': true_flag
            }, ensure_ascii=False)
            f_output.write(json_line + '\n')  # 每个 JSON 对象占一行

    # 在每个批次之间加入延迟
    time.sleep(0.05)  # 延迟 1 秒，可以根据需要调整

# 6. 计算准确率
def compute_accuracy(predictions, labels):
    return np.mean(np.array(predictions) == np.array(labels))

accuracy = compute_accuracy(predicted_classes, true_flags)
print(f"Accuracy: {accuracy * 100:.2f}%")

# 将准确率也写入文件
with open(output_file, 'a', encoding='utf-8') as f_output:
    json_line = json.dumps({'Accuracy': f"{accuracy * 100:.2f}%"}, ensure_ascii=False)
    f_output.write(json_line + '\n')



100%|██████████| 1133/1133 [1:48:18<00:00,  5.74s/it]

Accuracy: 0.00%



C:\Users\Administrator\AppData\Local\Temp\ipykernel_38956\2121505072.py:81: DeprecationWarning: elementwise comparison failed; this will raise an error in the future.
  return np.mean(np.array(predictions) == np.array(labels))


In [8]:
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score


# 6. 计算混淆矩阵及其他评估指标
def compute_metrics(predictions, labels):
    # 计算混淆矩阵
    cm = confusion_matrix(labels, predictions)

    # 计算各项指标
    accuracy = accuracy_score(labels, predictions)
    precision = precision_score(labels, predictions, average='weighted')  # 处理多分类任务，使用 weighted 平均
    recall = recall_score(labels, predictions, average='weighted')
    f1 = f1_score(labels, predictions, average='weighted')

    return cm, accuracy, precision, recall, f1

# 计算各项指标
cm, accuracy, precision, recall, f1 = compute_metrics(predicted_classes, true_flags)

# 打印评估结果
print(f"Confusion Matrix:\n{cm}")
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision: {precision * 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1 Score: {f1 * 100:.2f}%")

ValueError: Found input variables with inconsistent numbers of samples: [17579, 0]

In [11]:
# 文件路径
input_file = 'D:/PycharmProjects/TDFilter/DataSet/TrainData/train_confidence.jsonl'
output_file = 'D:/PycharmProjects/TDFilter/DataSet/TrainData/train_confidence_trimmed.jsonl'

# 行数限制
max_lines = 13535

# 读取文件并写入新的文件（只保留前 13535 行）
with open(input_file, 'r', encoding='utf-8') as infile, open(output_file, 'w', encoding='utf-8') as outfile:
    for i, line in enumerate(infile):
        if i < max_lines:
            outfile.write(line)
        else:
            break  # 超过行数后停止处理

print(f"文件处理完成，只保留前 {max_lines} 行。输出文件为: {output_file}")


文件处理完成，只保留前 13535 行。输出文件为: D:/PycharmProjects/TDFilter/DataSet/TrainData/train_confidence_trimmed.jsonl


In [3]:
import json
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score

# 文件路径
input_file = r'D:\PycharmProjects\TDFilter\experiment5\qwen\biological\iterate\results.jsonl'

# 初始化标签列表
true_flags = []
predicted_flags = []

# 读取 JSON Lines 文件并提取 true_flag 和 confidence_flag
with open(input_file, 'r', encoding='utf-8') as f:
    for line in f:
        data = json.loads(line.strip())
        true_flags.append(data['flag'])
        predicted_flags.append(data['final_flag'])  # 使用 'confidence_flag' 作为预测标签
#
# with open(input_file, 'r', encoding='utf-8') as f:
#     data = json.loads(f.read())  # 读取整个文件并解析为 JSON 对象
#
# true_flags = [item['flag'] for item in data]
# predicted_flags = [item['final_flag'] for item in data]  # 使用 'confidence_flag' 作为预测标签

# 计算混淆矩阵
cm = confusion_matrix(true_flags, predicted_flags)

# 计算准确率、精确率、召回率和 F1 分数
accuracy = accuracy_score(true_flags, predicted_flags)
precision = precision_score(true_flags, predicted_flags)  # 使用加权平均
recall = recall_score(true_flags, predicted_flags)        # 使用加权平均
f1 = f1_score(true_flags, predicted_flags)                # 使用加权平均

# 打印混淆矩阵和各项指标
print("Confusion Matrix:")
print(cm)
print(f"Accuracy: {accuracy * 100:.2f}%")
print(f"Precision: {precision* 100:.2f}%")
print(f"Recall: {recall * 100:.2f}%")
print(f"F1 Score: {f1 * 100:.2f}%")

# 如果需要将结果写入文件，你可以这样做：
output_file = r'D:\PycharmProjects\TDFilter\experiment5\qwen\biological\metrics_results.json'

# 将结果写入 JSON 文件
metrics_data = {
    'Confusion Matrix': cm.tolist(),  # 将 NumPy 数组转换为列表以存储为 JSON
    'Accuracy': f"{accuracy * 100:.2f}%",
    'Precision': f"{precision * 100:.2f}%",
    'Recall': f"{recall * 100:.2f}%",
    'F1 Score': f"{f1 * 100:.2f}%",
}

with open(output_file, 'w', encoding='utf-8') as f_output:
    json.dump(metrics_data, f_output, ensure_ascii=False, indent=4)

print(f"Metrics results saved to {output_file}")
# [[TN, FP],
#  [FN, TP]]


Confusion Matrix:
[[ 8138  7503]
 [ 5825 14764]]
Accuracy: 63.21%
Precision: 66.30%
Recall: 71.71%
F1 Score: 68.90%
Metrics results saved to D:\PycharmProjects\TDFilter\experiment5\qwen\biological\metrics_results.json


In [11]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.pyplot as plt
import json
from tqdm import tqdm

# 文件路径
file_path = r"D:\PycharmProjects\TDFilter\experiment5\qwen\radiation\basic\results.jsonl"
# 假设你的数据
with open(file_path,'r',encoding='utf-8') as f:
    data = json.load(f)
# 加载sentence-transformer模型
model = SentenceTransformer('sentence-transformers/all-MiniLM-L6-v2')

# 提取generateKnowledge字段并进行编码
generate_knowledge_list = [item['knowledge'] for item in data]
embeddings = model.encode(generate_knowledge_list)

# 计算每条数据的余弦相似度并找到matchingKnowledge
for i, embedding in tqdm(enumerate(embeddings), total=len(embeddings)):
    similarities = cosine_similarity([embedding], embeddings)[0]

    # 初始化最相似的generateKnowledge
    max_sim = -1
    matching_knowledge = None

    # 遍历所有其他generateKnowledge，寻找flag == 1且与当前embedding最相似的
    for j in range(len(similarities)):
        if i != j and data[j]["flag"] == 1:  # 排除自己且要求预测flag == 1
            if similarities[j] > max_sim:
                max_sim = similarities[j]
                matching_knowledge = data[j]['knowledge']

    # 为当前的data项增加matchingKnowledge字段
    data[i]['matchingKnowledge'] = matching_knowledge
    data[i]['similarity'] = float(max_sim)

# 将修改后的数据保存回文件
output_file_path = r"D:\PycharmProjects\TDFilter\experiment5\qwen\radiation\basic\results_processed.jsonl"
with open(output_file_path, 'w', encoding='utf-8') as f_out:
    json.dump(data, f_out, ensure_ascii=False, indent=4)

print("JSON 文件已更新并保存为:", output_file_path)


100%|██████████| 72460/72460 [1:11:11<00:00, 16.97it/s]


JSON 文件已更新并保存为: D:\PycharmProjects\TDFilter\experiment5\qwen\radiation\basic\results_processed.jsonl


In [4]:
# 查找description 并加入
import json
import re
# 输入和输出文件路径
input_file_path = r'D:\PycharmProjects\TDFilter\experiment5\qwen\biological\iterate\iterater.jsonl'
output_file_path = r'D:\PycharmProjects\TDFilter\experiment5\qwen\biological\iterate\iterater_preprocess.jsonl'

# 读取文件并处理
with open(input_file_path, 'r', encoding='utf-8') as infile, open(output_file_path, 'w', encoding='utf-8') as outfile:
    result = []
    for line in infile:
        data = json.loads(line.strip())  # 加载每行数据为 JSON 对象

        description_match = re.search(r'Description:\s*(.*?)(\.\n|$)', data['reasonable_prompt'], re.DOTALL)
        # 提取 description 并存为 data 的一个属性
        data["generateKnowledge"] = description_match.group(1).strip()  # 添加新的属性

        result.append(data)

    # 将处理后的数据写入新文件
    outfile.write(json.dumps(result, ensure_ascii=False))
print(f"处理完成，已将结果保存到: {output_file_path}")


处理完成，已将结果保存到: D:\PycharmProjects\TDFilter\experiment5\gpt35\Biological\basic\processed_results.jsonl


In [2]:

# 查找description 并加入
import json
import re

# 输入和输出文件路径
input_file_path = r'D:\PycharmProjects\TDFilter\experiment5\qwen\biological\iterate\iterater.jsonl'
with open(input_file_path, 'r', encoding='utf-8') as infile:
     json_data = json.load(infile)

print(len(json_data))


2041


In [2]:
import json
from sklearn.metrics import confusion_matrix
from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score, f1_score
# 输入文件路径
input_file_path = r'D:\PycharmProjects\TDFilter\experiment5\qwen\radiation\fake\results.jsonl'
json_data = []
# 读取 JSON 文件
limit=72459
with open(input_file_path, 'r', encoding='utf-8') as infile:
    for i,line in enumerate(infile):
        if(i>=limit):
            break
        data = json.loads(line)
        json_data.append(data)

# 提取 flag 和 confidence_flag 列表
flags = [item['flag'] for item in json_data]
confidence_flags = [item['final_flag'] for item in json_data]
# descriptions = [item.get('description', '') for item in json_data]  # 获取 description，若无则为空字符串

# 计算混淆矩阵
conf_matrix = confusion_matrix(flags, confidence_flags)
# 计算准确率、精确率、召回率和 F1 分数
accuracy = accuracy_score(flags, confidence_flags)
precision = precision_score(flags, confidence_flags)  # 使用加权平均
recall = recall_score(flags, confidence_flags)        # 使用加权平均
f1 = f1_score(flags, confidence_flags)                # 使用加权平均

# 打印混淆矩阵和各项指标
print("Confusion Matrix:")
print(conf_matrix)
print(f"Accuracy: {accuracy :.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall: {recall :.4f}")
print(f"F1 Score: {f1 :.4f}")

# 打印结果
# print("Descriptions:", descriptions[:5])  # 显示前5个描述字段



A module that was compiled using NumPy 1.x cannot be run in
NumPy 2.0.2 as it may crash. To support both 1.x and 2.x
versions of NumPy, modules must be compiled with NumPy 2.0.
Some module may need to rebuild instead e.g. with 'pybind11>=2.12'.

If you are a user of the module, the easiest solution will be to
downgrade to 'numpy<2' or try to upgrade the affected module.
We expect that some modules will need time to support NumPy 2.

Traceback (most recent call last):  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python39\lib\runpy.py", line 197, in _run_module_as_main
    return _run_code(code, main_globals, None,
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python39\lib\runpy.py", line 87, in _run_code
    exec(code, run_globals)
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Python39\lib\site-packages\ipykernel_launcher.py", line 17, in <module>
    app.launch_new_instance()
  File "C:\Users\Administrator\AppData\Local\Programs\Python\Pyth

AttributeError: _ARRAY_API not found

ImportError: numpy.core.multiarray failed to import

In [1]:
import json
from sklearn.metrics import confusion_matrix

# 输入文件路径
input_file_path = r'D:\PycharmProjects\TDFilter\experiment5\roberta\Biological\fake\results.jsonl'

# 读取 JSONL 文件
flags = []
confidence_flags = []
descriptions = []

with open(input_file_path, 'r', encoding='utf-8') as infile:
    # 因为是 JSONL 文件，每行是一个 JSON 对象，逐行读取并解析
    for line in infile:
        data = json.loads(line)
        flags.append(data['flag'])  # 真实值
        confidence_flags.append(data['final_flag'])  # 预测值
        descriptions.append(data.get('description', ''))  # 获取 description，若无则为空字符串

# 计算混淆矩阵
conf_matrix = confusion_matrix(flags, confidence_flags)

# 打印结果
print("Confusion Matrix:")
print(conf_matrix)
print("Number of records:", len(flags))
print("Descriptions:", descriptions[:5])  # 显示前5个描述字段


Confusion Matrix:
[[12068  4411]
 [ 1389 20327]]
Number of records: 38195
Descriptions: ['', '', '', '', '']


In [16]:
54827+8289-59172

3944

In [17]:
5816+3574-5248

4142